# 01b — NYC Taxi end-to-end (second dataset)

Demonstrates the same Fabric → AML → registered model flow on a
**different, larger, more realistic** dataset:
**NYC yellow taxi trip records** (`nyctaxi_yellow` in the lakehouse).

**Pre-req:** the `nyctaxi_yellow` Delta table must exist under
`Tables/dbo/` in the Fabric lakehouse, and `.env` must contain
`ONELAKE_TABLE_TAXI=nyctaxi_yellow`. See
[docs/09-add-nyctaxi-to-fabric.md](../docs/09-add-nyctaxi-to-fabric.md).

**Flow:**

1. Read taxi data from OneLake.
2. Quick EDA + sample down to 200K rows for fast POC iteration.
3. Engineer the binary target `tipped = (tipAmount > 0)`.
4. Save a local parquet snapshot.
5. Register the snapshot as an AML data asset (`contoso-poc-taxi-dataset`).
6. Submit a training job to `cpu-cluster` reusing `src.train`.
7. Register the trained MLflow model as `contoso-poc-taxi-model`.

This notebook **does not deploy** an endpoint — see the runbook for
the optional `deploy` step against the existing helper.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(pathlib.Path.cwd().parent / ".env", encoding="utf-8", override=True)

from src.config import load_settings

TAXI_TABLE      = os.getenv('ONELAKE_TABLE_TAXI', 'nyctaxi_yellow')
ASSET_NAME      = 'contoso-poc-taxi-dataset'
MODEL_NAME      = 'contoso-poc-taxi-model'
TARGET          = 'tipped'
DATA_VERSION    = '1'                 # bump after re-registering
COMPUTE         = 'cpu-cluster'
SAMPLE_ROWS     = 200_000             # downsample for POC speed
LOCAL_PARQUET   = '../data/local/nyctaxi_clf.parquet'

# Columns to drop before training — leakage or unhelpful identifiers
DROP_COLS = [
    'tipAmount',          # leakage: target derived from this
    'totalAmount',        # leakage: includes tip
    'paymentType',        # cash trips report tip=0 by convention -> trivial
    'puLocationId',
    'doLocationId',
    'storeAndFwdFlag',
]

s = load_settings()
print('Workspace: ', s.aml_workspace)
print('Taxi URI:  ', s.onelake_table_uri_for(table=TAXI_TABLE))

## 1. Read taxi data from OneLake + quick EDA

In [ ]:
from src.data import read_delta_table

df = read_delta_table(s, table=TAXI_TABLE)
print(f'Rows: {len(df):,}   Columns: {df.shape[1]}')
df.head(5)

In [ ]:
df.dtypes.to_frame('dtype')

## 2. Sample down + engineer `tipped` target

Full table is millions of rows. For a POC we sample 200K so the
training job finishes in a few minutes on `Standard_DS3_v2`. The
binary target is `tipped = tipAmount > 0`.

In [ ]:
import pandas as pd
import pathlib

if 'tipAmount' not in df.columns:
    raise RuntimeError(f'Expected tipAmount column. Got: {list(df.columns)}')

df = df.dropna(subset=['tipAmount']).copy()
df[TARGET] = (df['tipAmount'] > 0).astype(str)  # 'True' / 'False' -> sklearn-friendly

if len(df) > SAMPLE_ROWS:
    df = df.sample(n=SAMPLE_ROWS, random_state=42).reset_index(drop=True)

print(f'Sampled rows: {len(df):,}')
print('Target distribution:')
print(df[TARGET].value_counts(normalize=True).round(3))

In [ ]:
out = pathlib.Path(LOCAL_PARQUET)
out.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out, index=False)
print(f'Wrote {out.resolve()}  ({out.stat().st_size / 1e6:.1f} MB)')

## 3. Local smoke test — train a small model on the snapshot

In [ ]:
from src.train import train

metrics = train(
    data_path=LOCAL_PARQUET,
    target=TARGET,
    drop_cols=DROP_COLS,
    output_dir='../outputs/taxi_model',
)
metrics

## 4. Register the snapshot as an AML data asset

Uses the existing `copy_fabric_table_to_aml` helper, now
parameterized by `table=`.

In [ ]:
# Re-read from the local parquet so the snapshot exactly matches what
# the local smoke test trained on (sampled + with `tipped` already
# computed). The helper expects a OneLake table; for the engineered
# version we go through the AML SDK directly.
import tempfile
from pathlib import Path
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import DefaultAzureCredential

MLTABLE_YAML = '''$schema: https://azuremlschemas.azureedge.net/latest/MLTable.schema.json
type: mltable
paths:
  - file: ./data.parquet
transformations:
  - read_parquet
'''

ml = MLClient(DefaultAzureCredential(), s.subscription_id, s.resource_group, s.aml_workspace)

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    df.to_parquet(tmp_path / 'data.parquet', index=False)
    (tmp_path / 'MLTable').write_text(MLTABLE_YAML)
    asset = ml.data.create_or_update(Data(
        name=ASSET_NAME,
        description='NYC taxi sample (200K rows) with engineered `tipped` target.',
        path=str(tmp_path),
        type=AssetTypes.MLTABLE,
    ))
DATA_VERSION = asset.version
print(f'Registered: azureml:{asset.name}:{DATA_VERSION}')

## 5. Submit remote training job to `cpu-cluster`

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import InputOutputModes

job = command(
    code='..',
    command=(
        'python -m src.train '
        '--data-path ${{inputs.training_data}} '
        f'--target {TARGET} '
        f'--drop-cols {" ".join(DROP_COLS)} '
        '--output-dir outputs/model'
    ),
    inputs={
        'training_data': Input(
            type=AssetTypes.MLTABLE,
            path=f'azureml:{ASSET_NAME}:{DATA_VERSION}',
            mode=InputOutputModes.RO_MOUNT,
        ),
    },
    environment='azureml://registries/azureml/environments/sklearn-1.5/labels/latest',
    compute=COMPUTE,
    display_name='contoso-poc-taxi-training',
    experiment_name='contoso-poc-taxi',
)
returned_job = ml.jobs.create_or_update(job)
print('Job:    ', returned_job.name)
print('Studio: ', returned_job.studio_url)

In [ ]:
ml.jobs.stream(returned_job.name)
final = ml.jobs.get(returned_job.name)
print('Status:', final.status)

## 6. Register the trained MLflow model

In [ ]:
from azure.ai.ml.entities import Model

model = Model(
    name=MODEL_NAME,
    path=f'azureml://jobs/{returned_job.name}/outputs/artifacts/paths/outputs/model/',
    type=AssetTypes.MLFLOW_MODEL,
    description='RandomForest classifier predicting whether an NYC yellow-taxi trip is tipped.',
)
registered = ml.models.create_or_update(model)
print(f'Registered: {registered.name} v{registered.version}')
print('\nNext (optional): deploy to its own endpoint with')
print(f'  uv run python -m src.deploy_online_endpoint create-endpoint --endpoint-name contoso-taxi-endpoint')
print(f'  uv run python -m src.deploy_online_endpoint grant-rbac     --endpoint-name contoso-taxi-endpoint')
print(f'  uv run python -m src.patch_mlflow_model --name {MODEL_NAME} --version {registered.version} --add-package azureml-ai-monitoring --description "v2: monitoring"')
print(f'  uv run python -m src.deploy_online_endpoint deploy --endpoint-name contoso-taxi-endpoint --model-name {MODEL_NAME} --model-version <new-version>')